In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from PIL import Image
import os
import albumentations as A
from albumentations.pytorch import ToTensorV2
import timm

print("albumentations version:", albumentations.__version__ if (albumentations := __import__("albumentations")) else None)

# --- Load CSV ---
csv_path = "/kaggle/input/datasets/anishapanja/bc-xai-patches-split-2000/patches_with_split_2000.csv"
df = pd.read_csv(csv_path)
print("Shape:", df.shape)
print("Subtype counts:\n", df["subtype_clean"].value_counts())
print("Split counts:\n", df["split"].value_counts())

# --- Verify image paths open ---
row_a2 = df[df["institution"] == "A2"].iloc[0]
row_e2 = df[df["institution"] == "E2"].iloc[0]
for label, row in [("A2", row_a2), ("E2", row_e2)]:
    path = row["patch_path"]
    exists = os.path.exists(path)
    print(f"{label} | exists: {exists}")
    if exists:
        img = Image.open(path)
        print(f"   image size: {img.size}, mode: {img.mode}")

# --- Label map ---
LABEL_MAP = {"LumA": 0, "LumB": 1, "Her2": 2, "Basal": 3}

# --- Transforms ---
train_transform = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.05, hue=0.02, p=0.3),
    A.GaussianBlur(blur_limit=(3, 5), p=0.1),
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

eval_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2(),
])

# --- Dataset class ---
class PatchDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = Image.open(row["patch_path"]).convert("RGB")
        img = np.array(img)
        img = self.transform(image=img)["image"]
        label = LABEL_MAP[row["subtype_clean"]]
        return img, label

# --- Splits ---
train_df = df[df["split"] == "train"].reset_index(drop=True)
val_df = df[df["split"] == "val"].reset_index(drop=True)
test_df = df[df["split"] == "test"].reset_index(drop=True)
print("\ntrain_df:", len(train_df), "val_df:", len(val_df), "test_df:", len(test_df))

train_dataset = PatchDataset(train_df, train_transform)
val_dataset = PatchDataset(val_df, eval_transform)
test_dataset = PatchDataset(test_df, eval_transform)

# Sanity check one real sample
img, label = train_dataset[0]
print("Sample 0 shape:", img.shape, "dtype:", img.dtype, "label:", label)

# --- Class weights (SAFE: built via reindex, not positional sort) ---
class_counts = train_df["subtype_clean"].map(LABEL_MAP).value_counts().reindex([0, 1, 2, 3])
print("\nTrain patch counts per class (0=LumA,1=LumB,2=Her2,3=Basal):")
print(class_counts)
assert list(class_counts.index) == [0, 1, 2, 3], "Class count index is not in expected order!"
assert class_counts.isna().sum() == 0, "A class is missing from train split entirely!"

class_weights = 1.0 / class_counts.values
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32)
print("Class weights (for loss):", class_weights_tensor)

# --- Sampler ---
train_labels = train_df["subtype_clean"].map(LABEL_MAP).values
sample_weights = class_weights[train_labels]
sample_weights_tensor = torch.tensor(sample_weights, dtype=torch.float32)

sampler = WeightedRandomSampler(
    weights=sample_weights_tensor,
    num_samples=len(sample_weights_tensor),
    replacement=True
)

# --- DataLoaders ---
BATCH_SIZE = 32
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, sampler=sampler, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
print("\ntrain_loader batches:", len(train_loader))
print("val_loader batches:", len(val_loader))
print("test_loader batches:", len(test_loader))

# --- Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("\nDevice:", device)

# --- Model: EfficientNet-B4 ---
model_effnet = timm.create_model("tf_efficientnet_b4", pretrained=True, num_classes=4)
model_effnet = model_effnet.to(device)
print("\nEfficientNet-B4 loaded. Total params:", sum(p.numel() for p in model_effnet.parameters()))

# --- Optimizer, scheduler, loss for EfficientNet-B4 ---
optimizer_effnet = torch.optim.AdamW(model_effnet.parameters(), lr=1e-4, weight_decay=1e-4)
scheduler_effnet = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_effnet, T_max=20, eta_min=1e-6)
criterion = nn.CrossEntropyLoss(weight=class_weights_tensor.to(device))

print("\nOptimizer, scheduler, loss function ready.")
print("Setup complete — ready for training loop.")

albumentations version: 2.0.8
Shape: (308844, 6)
Subtype counts:
 subtype_clean
LumA     160941
Basal     68722
LumB      56214
Her2      22967
Name: count, dtype: int64
Split counts:
 split
train    219751
val       44731
test      44362
Name: count, dtype: int64
A2 | exists: True
   image size: (256, 256), mode: RGB
E2 | exists: True
   image size: (256, 256), mode: RGB

train_df: 219751 val_df: 44731 test_df: 44362
Sample 0 shape: torch.Size([3, 224, 224]) dtype: torch.float32 label: 2

Train patch counts per class (0=LumA,1=LumB,2=Her2,3=Basal):
subtype_clean
0    115994
1     40842
2     15905
3     47010
Name: count, dtype: int64
Class weights (for loss): tensor([8.6211e-06, 2.4485e-05, 6.2873e-05, 2.1272e-05])

train_loader batches: 6868
val_loader batches: 1398
test_loader batches: 1387

Device: cuda


model.safetensors:   0%|          | 0.00/77.9M [00:00<?, ?B/s]


EfficientNet-B4 loaded. Total params: 17555788

Optimizer, scheduler, loss function ready.
Setup complete — ready for training loop.
